<br>

<center><img src='../Header_image/header_image.png' style="width:900px"></center>

<br>

<p style="font-size: 30px; text-align: center;">
<b>✏️ Homework Session 9 — Seismic I </b></p>

---

## Objective

In this homework you will build a synthetic seismogram workflow using the **B-41 well**
and explore a 3-D seismic volume.

You will:

1. Load a LAS file and extract log curves in SI units.
2. Correct for the shallow section (water column + replacement interval).
3. Despike the density and sonic logs.
4. Compute acoustic impedance and the reflection coefficient series. 
5. Load a 3-D SEG-Y seismic volume and plot three orthogonal slices.

4.5 (Bonus question) Build a synthetic seismogram by convolving the reflectivity with a Ricker wavelet. 

### Input files (all inside `./Input/`)

| File | Description |
|------|-------------|
| `B-41.las` | B-41 well — sonic (DT) and density (RHOB) logs |
| `F3_Dip_steered_median_subvolume_IL230-430_XL475-675_T1600-1800.sgy` | F3 seismic sub-volume (SEG-Y) |

---

### Setup — run this cell first

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import lasio
import segyio

print("Libraries loaded ✓")

Libraries loaded ✓


---
## Part 1 — Load the LAS File and Convert Units

A LAS (Log ASCII Standard) file stores well-log curves and well metadata.
The `lasio` library parses them into a `LASFile` object:

```python
import lasio
well = lasio.read('./Input/B-41.las')
print(well.curves)          # list available curves
df = well.df().reset_index()  # convert to DataFrame
```

All depth and elevation values in the LAS header are stored in **feet**.
We will convert everything to SI units (metres, kg/m³, μs/m).

| Variable | Source column | Conversion |
|----------|--------------|------------|
| `DEPTH` | `DEPTH:1` (ft) | ÷ 3.28084 → m |
| `DT` | `DT` (μs/ft) | × 3.28084 → μs/m |
| `RHOB` | `RHOB` (g/cm³) | × 1000 → kg/m³ |


<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise 1 — Load the LAS file and extract log curves</b>

**1a.** Load `./Input/B-41.las` into a variable called `well`.  
Print the list of available curves and display the first 10 rows of the DataFrame.

**1b.** Write a function `ft2m(x)` that converts feet to metres.  
It should handle both scalars and array-like inputs (use `try / except TypeError`).

**1c.** Extract `DEPTH`, `DT`, and `RHOB` from the DataFrame, applying the unit
conversions shown in the table above. Print the depth range and the min/max of
each curve.
</div>

In [2]:
# ── 1a: Load LAS file ─────────────────────────────────────────────────────────
# YOUR CODE HERE


# ── 1b: Unit-conversion helper ────────────────────────────────────────────────
def ft2m(x):
    # YOUR CODE HERE
    pass


# ── 1c: Extract and convert curves ───────────────────────────────────────────
# YOUR CODE HERE


---
## Part 2 — Shallow-Section Time Correction

The sonic log does not start at the surface. We need to estimate the
**two-way travel time (TWT) at the top of the log** using replacement velocities
for the water column and the soft-sediment interval above the log.

```
Sea level (MSL)
│
├──── water column ──────────  V_water = 1 480 m/s
├──── replacement interval ──  V_repl  = 1 600 m/s
└──── top of sonic log  ─────  → LOGSS_TWT  ← this is what we need
```

The well datums live in the LAS header (all in feet):

```python
KB    = well.well['KB'].value    # Kelly Bushing elevation
WD    = well.well['GL'].value    # Water depth below MSL
LOGSS = well.well['STRT'].value  # Top of log below KB
```

The TWT at the log top is computed step by step:

$$KB_{TWT} = \frac{2\,|KB_m|}{V_{water}}, \quad
W_{TWT} = KB_{TWT} + \frac{2\,|WD_m|}{V_{water}}$$

$$REPI = LOGSS_m - KB_m + WD_m, \quad
REPI_{TWT} = \frac{2 \cdot REPI}{V_{repl}}$$

$$LOGSS_{TWT} = W_{TWT} + REPI_{TWT}$$


<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise 2 — Compute the log-start TWT</b>

**2a.** Extract `KB`, `WD`, and `LOGSS` from `well.well` and convert them to metres.

**2b.** Using `V_water = 1480 m/s` and `V_repl = 1600 m/s`, compute
`KB_TWT`, `W_TWT`, `REPI`, `REPI_TWT`, and `LOGSS_TWT` following the formulas above.

**2c.** Print a summary of all six quantities with their values and units.

> **Hint:** `LOGSS_TWT` should be somewhere between 0.3 s and 0.7 s.
> If your value looks very different, check that you're taking the absolute value
> of `KB` and `WD` before dividing.
</div>

In [3]:
# ── 2a: Extract well datums ───────────────────────────────────────────────────
V_water = 1480   # [m/s]
V_repl  = 1600   # [m/s]

# YOUR CODE HERE


# ── 2b: TWT corrections ───────────────────────────────────────────────────────
# YOUR CODE HERE


# ── 2c: Summary ───────────────────────────────────────────────────────────────
# YOUR CODE HERE


---
## Part 3 — Log Quality Control: Despiking

Raw well logs often contain high-amplitude **spikes** caused by tool noise or borehole
washouts. We remove them by comparing each sample to a rolling median: if the deviation
exceeds a threshold, the sample is clamped.

**Step 1 — rolling median** using NumPy stride tricks:

```python
def rolling_window(a, window):
    shape   = a.shape[:-1] + (a.shape[-1] - window + 1, window)
    strides = a.strides + (a.strides[-1],)
    return np.lib.stride_tricks.as_strided(a, shape=shape, strides=strides)

# Usage (window = 13 samples ≈ 2 m at 0.1524 m/sample):
sm = np.median(rolling_window(arr, 13), axis=-1)
sm = np.pad(sm, 13 // 2, mode='edge')   # restore original length
```

> **Tip:** Pass `arr.to_numpy()` when `arr` is a Pandas Series —
> stride tricks requires a plain NumPy array.

**Step 2 — clamp spikes:**

```python
def despike(curve, curve_sm, max_clip):
    curve, curve_sm = np.asarray(curve), np.asarray(curve_sm)
    out = np.copy(curve)
    out[curve - curve_sm >  max_clip] = curve_sm[curve - curve_sm >  max_clip] + max_clip
    out[curve_sm - curve >  max_clip] = curve_sm[curve_sm - curve >  max_clip] - max_clip
    return out
```


<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise 3 — Despike the density and sonic logs</b>

**3a.** Copy both functions (`rolling_window` and `despike`) into the cell below.

**3b.** Apply them to both logs using the parameters below.
Store results as `RHOB_DESP` and `DT_DESP`.

| Log | `max_clip` |
|-----|-----------|
| `RHOB` | 100 kg/m³ |
| `DT` | 10 μs/m |

Use a window of **13 samples** for both.

**3c.** For each log, make a plot over the **full depth range** showing three lines:
Original (black), Smooth (blue), Despiked (red).
Add a title, axis labels, and a legend.
</div>

In [4]:
# ── 3a: Helper functions ──────────────────────────────────────────────────────
# YOUR CODE HERE


# ── 3b: Apply despiking ───────────────────────────────────────────────────────
WINDOW = 13
# YOUR CODE HERE


# ── 3c: Diagnostic plots ─────────────────────────────────────────────────────
# YOUR CODE HERE


---
## Part 4 — Acoustic Impedance and Reflection Coefficient Series

With clean logs we can compute the two key quantities for a synthetic seismogram.

### Time-to-depth relationship (TDR)

Integrating the scaled sonic log gives cumulative two-way travel time along the well:

$$TDR(z) = LOGSS_{TWT} + 2 \sum_{i} \Delta t_i \cdot \Delta z$$

where $\Delta z = 0.1524$ m (6-inch sample interval) and $\Delta t$ is slowness [s/m].

```python
DT_SCALED = 0.1524 * np.nan_to_num(DT) / 1e6   # [s/sample]
TCUM      = 2.0 * np.cumsum(DT_SCALED)
TDR       = LOGSS_TWT + TCUM
```

### Acoustic impedance

$$Z = V_P \times \rho = \frac{10^6}{DT_{\text{desp}}} \times RHOB_{\text{desp}}
\quad [\text{kg}\,\text{m}^{-2}\text{s}^{-1}]$$

### Reflection coefficient series

$$RC_i = \frac{Z_{i+1} - Z_i}{Z_{i+1} + Z_i}$$

Note: `RC` has one fewer element than `Z` and corresponds to `DEPTH[:-1]`.


<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise 4 — Compute impedance, TDR and reflectivity</b>

**4a.** Compute `DT_SCALED`, `TCUM`, and `TDR` using the despiked DT log.

**4b.** Compute the acoustic impedance `Z` using the despiked logs.  
Print the min, max, and mean of `Z`.

**4c.** Compute the reflection coefficient series `RC`.  
Print how many non-zero coefficients there are and the max absolute value.

**4d.** Produce a **four-panel figure** with a shared depth axis (y-axis inverted,
0–4500 m), one panel per quantity:

| Panel | X-axis | Colour |
|-------|--------|--------|
| (a) Sonic log `DT` | μs/m | steelblue |
| (b) Cumul. TWT `TCUM` | s | steelblue |
| (c) Impedance `Z` | kg m⁻²s⁻¹ | black |
| (d) Reflectivity `RC` vs `DEPTH[:-1]` | — | black |

Add a title to each panel and label the x-axes.  
*(You do not need to annotate formation tops.)*
</div>

In [5]:
# ── 4a: Time-to-depth relationship ───────────────────────────────────────────
# YOUR CODE HERE


# ── 4b: Acoustic impedance ────────────────────────────────────────────────────
# YOUR CODE HERE


# ── 4c: Reflection coefficient series ────────────────────────────────────────
# YOUR CODE HERE


# ── 4d: Four-panel figure ─────────────────────────────────────────────────────
# YOUR CODE HERE


---
## Part 4.5 *(Bonus)* — Synthetic Seismogram Creation

To produce the synthetic seismogram, we convolve the reflectivity series with a seismic wavelet.

The seismic data is sampled at **4 ms** on a regular time grid.
We interpolate the impedance log from the irregular TDR grid onto this regular grid using
`np.interp`, then re-derive the reflection coefficients at seismic sample rate.

```python
dt_seis = 0.004              # seismic sample interval [s]
t_max   = 3.0                # max TWT [s]
t       = np.arange(0.0, t_max, dt_seis)
Z_t     = np.interp(x=t, xp=TDR, fp=Z)          # impedance at seismic sample rate
RC_t    = (Z_t[1:] - Z_t[:-1]) / (Z_t[1:] + Z_t[:-1])
RC_t    = np.nan_to_num(RC_t)
```

#### Ricker wavelet

A zero-phase Ricker wavelet (dominant frequency $f$):

$$w(t) = \left(1 - 2\pi^2 f^2 t^2 \right) e^{-\pi^2 f^2 t^2}$$

```python
def ricker(f, length, dt):
    n = int(length / dt)
    t = np.linspace(-length / 2, (length - dt) / 2, n)
    w = (1.0 - 2.0 * (np.pi * f * t)**2) * np.exp(-(np.pi * f * t)**2)
    return t, w
```

#### Convolve to get the synthetic

```python
synth = np.convolve(RC_t, w, mode='same')
```
<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise Bonus 4.5 — Build and plot the synthetic seismogram</b>

**4.5a.** Using the code snippets above, compute `Z_t`, `RC_t`, and the time axis `t`.

**4.5b.** Copy the `ricker` function and generate a **25 Hz** wavelet with:
- `length = 0.512 s`
- `dt = 0.004 s`

Plot the wavelet (amplitude vs time in ms). Label axes.

**4.5c.** Convolve `RC_t` with your wavelet to produce `synth`.

**4.5d.** Produce the **four-panel time-domain figure** with:
- Panel (a): impedance `Z` vs `DEPTH` (depth domain — for reference)
- Panel (b): impedance `Z_t` vs `t` (time domain)
- Panel (c): reflectivity `RC_t` as a **stick plot** (`ax.hlines`)
- Panel (d): synthetic seismogram with positive amplitudes filled black

Annotate panels (a) with formation tops in depth, panels (b)–(d) with formation tops in TWT.

> **Tip:** Use `ax.fill_betweenx(t[:-1], synth, 0, where=synth > 0, color='k')` for the
> wiggle fill.

</div>

---
#### Part 5 — Loading and Visualising a 3-D Seismic Volume

We use the `segyio` library to read a SEG-Y file directly into a NumPy array:

```python
import segyio

filename = ('./Input/F3_Dip_steered_median_subvolume_'
            'IL230-430_XL475-675_T1600-1800.sgy')

with segyio.open(filename) as s:
    print(f"Inlines   : {s.ilines.min()} – {s.ilines.max()}")
    print(f"Crosslines: {s.xlines.min()} – {s.xlines.max()}")
    print(f"Samples   : {s.samples.size}")
    c = segyio.cube(s)          # shape: (n_inlines, n_xlines, n_samples)

c = c.astype(np.int16)          # cast to int16 to halve memory usage
```

The cube axes are `(inline_index, crossline_index, time_sample_index)` — all **0-based**.

To clip the colourscale symmetrically at the 98th percentile:

```python
ma = np.percentile(np.abs(section), 98)
ax.imshow(section.T, vmin=-ma, vmax=ma, cmap='seismic', aspect='auto')
```


<div class="alert alert-block alert-warning">
<font size="6">&#128675;</font> <b>Exercise 5 — Load the SEG-Y volume and plot three slices</b>

**5a.** Load the SEG-Y file into an array `c` (cast to `int16`).  
Print the cube shape, data type, and memory usage in MB.

**5b.** Extract the following three slices (indices are 0-based within the sub-volume):

| Slice | Index | Variable name |
|-------|-------|--------------|
| Inline | 15 | `il_slice` |
| Crossline | 75 | `xl_slice` |
| Time / depth sample | 40 | `ts_slice` |

**5c.** Plot all three slices in a **single figure with three subplots** side by side.

- **Inline and crossline** slices: use `cmap='seismic'`, clip at the 98th percentile,
  `aspect='auto'`, label axes (crossline index / inline index on x; time sample on y),
  invert the y-axis so time increases downward.
- **Time/depth slice**: use `cmap='RdYlBu_r'`, same amplitude clipping,
  label axes (crossline index on x, inline index on y).
- Give each subplot a descriptive title (e.g. `"Inline 15  (index 15)"`).
- Add a colourbar to each subplot.

</div>

In [6]:
# ── 5a: Load SEG-Y volume ─────────────────────────────────────────────────────
filename = ('./Input/F3_Dip_steered_median_subvolume_'
            'IL230-430_XL475-675_T1600-1800.sgy')

# YOUR CODE HERE


# ── 5b: Extract slices ────────────────────────────────────────────────────────
il_idx = 15    # inline index
xl_idx = 75    # crossline index
ts_idx = 40    # time/depth sample index

# YOUR CODE HERE


# ── 5c: Three-panel figure ────────────────────────────────────────────────────
# YOUR CODE HERE
